# 👥 W4: 채용공고 & 작업지시서 자동 생성
**hexa-1 — Week 4** | ⏱️ 60분 | 🎯 채용공고 + 작업지시서 각 3건

## Step 0: 환경 설정

In [ ]:
import subprocess
subprocess.run(['apt-get', '-qq', '-y', 'install', 'fonts-nanum'], capture_output=True)
!pip install -q google-generativeai pandas
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
fm._load_fontmanager(try_read_cache=False)
nanum = next((f.fname for f in fm.fontManager.ttflist if 'Nanum' in f.name), None)
if nanum: plt.rcParams['font.family'] = fm.FontProperties(fname=nanum).get_name()
plt.rcParams['axes.unicode_minus'] = False
try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
except:
    API_KEY = input('GEMINI_API_KEY 입력: ')
import google.generativeai as genai
genai.configure(api_key=API_KEY)
model = genai.GenerativeModel('gemini-2.0-flash')
print('✅ 환경 설정 완료')


## Step 1: 회사 & 문서 정보 입력 (✏️ 여기만 수정)

In [ ]:
COMPANY = {
    'name': '✏️ [교육 기업명]',
    'industry': '✏️ [업종/제품]',
    'location': '✏️ [지역]',
    'benefits': '4대보험, 식비지원, 연차'
}
JOB_OPENINGS = [
    {'position': 'AI 품질관리 담당자', 'type': 'job', 'experience': '3년 이상'},
    {'position': '생산라인 반장', 'type': 'job', 'experience': '5년 이상'},
]
WORK_ORDERS = [
    {'no': 'WO-2026-001', 'task': '3호 라인 월간 설비 점검', 'assignee': '이철수', 'deadline': '2026-03-10', 'safety': '보호장구 착용 필수'},
    {'no': 'WO-2026-002', 'task': '신제품 샘플 5개 가공', 'assignee': '박영희', 'deadline': '2026-03-07', 'safety': '절삭유 취급 주의'},
]
print(f'✅ 채용공고 {len(JOB_OPENINGS)}건, 작업지시서 {len(WORK_ORDERS)}건 준비')


## Step 2: 채용공고 자동 생성

In [ ]:
job_docs = {}
for job in JOB_OPENINGS:
    prompt = f"""제조업 HR 담당자로서 다음 채용공고를 작성해주세요.
회사: {COMPANY['name']} ({COMPANY['industry']}) / 위치: {COMPANY['location']}
채용 직무: {job['position']} / 경력: {job['experience']}
복리후생: {COMPANY['benefits']}
포함 항목: 회사 소개, 담당 업무, 자격 요건, 우대 사항, 복리후생, 지원 방법
형식: 마크다운, 전문적이고 매력적인 문체"""
    resp = model.generate_content(prompt)
    job_docs[job['position']] = resp.text
    print(f'✅ {job["position"]} 채용공고 생성 완료')

# 첫 번째 미리보기
first = list(job_docs.keys())[0]
print(f'\n=== {first} 미리보기 ===')
print(list(job_docs.values())[0][:600], '...')


## Step 3: 작업지시서 자동 생성

In [ ]:
work_docs = {}
for wo in WORK_ORDERS:
    prompt = f"""제조업 생산관리 담당자로서 작업지시서를 작성해주세요.
지시번호: {wo['no']} | 작업내용: {wo['task']}
담당자: {wo['assignee']} | 완료기한: {wo['deadline']}
안전수칙: {wo['safety']}
형식: 마크다운 표 포함, 서명란 포함, 공식적 문체"""
    resp = model.generate_content(prompt)
    work_docs[wo['no']] = resp.text
    print(f'✅ {wo["no"]} 작업지시서 생성 완료')


## Step 4: ZIP으로 일괄 다운로드

In [ ]:
import os, zipfile
os.makedirs('hr_docs', exist_ok=True)
for pos, content in job_docs.items():
    with open(f'hr_docs/채용공고_{pos.replace(" ","_")}.md', 'w', encoding='utf-8') as f:
        f.write(content)
for no, content in work_docs.items():
    with open(f'hr_docs/작업지시서_{no}.md', 'w', encoding='utf-8') as f:
        f.write(content)
with zipfile.ZipFile('hr_docs.zip', 'w') as zf:
    for root, dirs, files in os.walk('hr_docs'):
        for file in files: zf.write(os.path.join(root, file))
from google.colab import files
files.download('hr_docs.zip')
print(f'✅ {len(job_docs)+len(work_docs)}건 HR 문서 다운로드 완료!')


---
## 🔥 확장 과제
1. `JOB_OPENINGS`에 실제 채용 직무 추가
2. 영문 채용공고도 동시 생성 (`language='English'` 파라미터 추가)
3. `hr_doc_generator.py`를 CLI에서 실행해보기